In [ ]:
#install dependency
! pip install -q groq ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 71.6 MB/s eta 0:00:00


In [ ]:
# API Key Integrate
import os
from getpass import getpass
if not os.environ.get("GROQ_API_KEY"):
  os.environ["GROQ_API_KEY"]=getpass("Enter your key:")
  print("API key set."if os.environ.get("GROQ_API_KEY")else "No API Key found")

Enter your key:··········
API key set.


In [ ]:
from groq import Groq
client=Groq()
for m in client.models.list().data:
  print(m.id)

whisper-large-v3
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-arabic-saudi
whisper-large-v3-turbo
openai/gpt-oss-20b
openai/gpt-oss-120b
canopylabs/orpheus-v1-english
qwen/qwen3.8-27b
meta-llama/llama-prompt-guard-2-86m
allam-2-7b


In [ ]:
from groq import Groq
from dataclasses import dataclass, field
from typing import List, Dict

@dataclass
class Chatbot:
  model :str = "openai/gpt-oss-20b"
  system_prompt: str ="You are a helpful,concise assistant."
  temperature:float = 0.7
  max_tokens: int = 1024

  _client: Groq = field(default=None,repr=False)
  history: List[Dict[str,str]] = field(default_factory=list, repr = False)

  def __post_init__(self):
    self._client = Groq() # reads GROQ_API_KEY from env

  def send(self,user_message: str) -> str:
    self.history.append({"role":"user","content": user_message })
    # Prepare messages for API call, including system prompt at the beginning
    messages_for_api = [{"role":"system","content":self.system_prompt}] + self.history

    try:
      response = self._client.chat.completions.create(
          model = self.model,
          messages = messages_for_api,
          temperature = self.temperature,
          max_tokens = self.max_tokens,
      )
    except Exception as e:
      self.history.pop() # Remove the last user message if API call fails
      raise RuntimeError(f"Groq API error: {e}") from e

    reply = response.choices[0].message.content
    self.history.append({"role":"assistant","content":reply})
    return reply

In [ ]:
bot = Chatbot()
bot_msg = bot.send("tell me about AI")
print(bot_msg)

**Artificial Intelligence (AI)** – the science and engineering of making machines that can perform tasks normally requiring human intelligence.

| Aspect | What it means |
|--------|---------------|
| **Core goal** | Enable systems to perceive, reason, learn, and act autonomously. |
| **Key capabilities** | • Perception (vision, speech, sensors)  <br>• Reasoning & planning <br>• Learning from data <br>• Natural‑language understanding |
| **Types** | • **Narrow/Weak AI** – excels at a single task (e.g., image classification, game playing). <br>• **General AI** – would match or exceed human versatility (still theoretical). |
| **Common techniques** | • Machine learning (deep learning, reinforcement learning) <br>• Symbolic AI (rule‑based systems) <br>• Hybrid approaches |
| **Applications** | • Healthcare (diagnosis, drug discovery) <br>• Finance (fraud detection, algorithmic trading) <br>• Autonomous vehicles <br>• Personal assistants <br>• Robotics, manufacturing, smart homes |
| **Eth

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
import html
import traceback
bot = Chatbot(model="openai/gpt-oss-20b")   # default

model_dropdown = widgets.Dropdown(
    options=["openai/gpt-oss-20b", "openai/gpt-oss-120b", "qwen/qwen3.8-27b", "allam-2-7b"],
    value=bot.model,
    desdescriptioncription="Model:",
    layout=widgets.Layout(width="300px"),
)

temp_slider = widgets.FloatSlider(
    value=bot.temperature, min=0.0, max=1.0, step=0.1,
    description="Temp:", continuous_update=False,
    layout=widgets.Layout(width="300px"),
)

chat_log = widgets.Output(
    layout=widgets.Layout(border="1px solid #ccc", height="400px", overflow_y="auto", padding="8px")
)
text_input = widgets.Text(placeholder="Type a message and press Enter...", layout=widgets.Layout(width="80%"))
send_button = widgets.Button(description="Send", button_style="primary")
clear_button = widgets.Button(description="Clear chat", button_style="warning")
status_label = widgets.Label(value="")

input_row = widgets.HBox([text_input, send_button, clear_button])
controls_row = widgets.HBox([model_dropdown, temp_slider])
ui = widgets.VBox([controls_row, chat_log, input_row, status_label])


def render_bubble(role, text):
    if role == "user":
        bg, color, align = "#DCF8C6", "#000000", "right"
    elif role == "assistant":
        bg, color, align = "#F1F0F0", "#000000", "left"
    else:  # error
        bg, color, align = "#FFD6D6", "#7A0000", "left"

    safe_text = html.escape(text).replace("\n", "<br>")
    bubble = f"""
    <div style="text-align:{align}; margin:6px 0;">
      <span style="display:inline-block; background:{bg}; color:{color}; padding:8px 12px;
                    border-radius:10px; max-width:75%; text-align:left;">
        <b>{role}:</b><br>{safe_text}
      </span>
    </div>
    """
    with chat_log:
        display(HTML(bubble))


def on_send(_=None):
    message = text_input.value.strip()
    if not message:
        return
    text_input.value = ""
    text_input.disabled = send_button.disabled = True
    status_label.value = "Waiting for response..."
    render_bubble("user", message)

    bot.model = model_dropdown.value
    bot.temperature = temp_slider.value

    try:
        reply = bot.send(message)
        render_bubble("assistant", reply)
        status_label.value = ""
    except Exception as e:
        render_bubble("error", str(e))
        status_label.value = "Error — see message above."
        traceback.print_exc()
    finally:
        text_input.disabled = send_button.disabled = False


def on_clear(_=None):
    bot.reset()
    chat_log.clear_output()
    status_label.value = "Chat cleared."


send_button.on_click(on_send)
clear_button.on_click(on_clear)
text_input.on_submit(on_send)

display(ui)